# 14 EMA Momentum Sweep

Compare how FID and linear probe accuracy change with the teacher EMA momentum value.

If momentum is too high, the teacher becomes stale → the distillation signal weakens
If momentum is too low, the teacher becomes unstable → gradient noise increases

In [ ]:
import torch
from src.experiments import load_cfg, deep_update

# ── GPU Auto-Detection ───────────────────────────────────────
n_gpu = torch.cuda.device_count()
print(f"Detected number of GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")
if n_gpu == 0:
    print("  (No GPU — running on CPU)")


In [ ]:
from src.experiments import load_cfg, deep_update, launch_train
from pathlib import Path
import pandas as pd

base_cfg = load_cfg("configs/cifar10.yaml")
base_cfg = deep_update(base_cfg, {
    "wandb": {"enabled": True, "tags": ["cifar10", "ema_sweep"]},
})

momentum_values = [0.990, 0.996, 0.999]
rows = []

for m in momentum_values:
    cfg = deep_update(base_cfg, {"sdd": {"teacher_momentum": m}})
    cfg["run_name"] = f"cifar10_ema_{str(m).replace('.', '')}"
    print(f"\n▶ teacher_momentum={m}")
    launch_train(cfg, num_processes=None, with_eval=True)

    csv = Path(f"outputs/logs/{cfg['run_name']}_history.csv")
    if csv.exists():
        df = pd.read_csv(csv)
        last = df.dropna(subset=["val_fid"]).iloc[-1] if "val_fid" in df.columns else df.iloc[-1]
        rows.append({"teacher_momentum": m,
                     "fid": last.get("val_fid"),
                     "linear_probe_acc": last.get("val_linear_probe_acc")})

results = pd.DataFrame(rows)
results


In [ ]:
import matplotlib.pyplot as plt

if not results.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    x = results["teacher_momentum"].astype(str)

    axes[0].bar(x, results["fid"], color="steelblue")
    axes[0].set_xlabel("Teacher EMA momentum"); axes[0].set_ylabel("FID ↓")
    axes[0].set_title("FID vs EMA momentum")
    for i, v in enumerate(results["fid"]):
        if v is not None:
            axes[0].text(i, v + 0.3, f"{v:.1f}", ha="center", fontsize=10)

    axes[1].bar(x, results["linear_probe_acc"], color="darkorange")
    axes[1].set_xlabel("Teacher EMA momentum"); axes[1].set_ylabel("Probe acc ↑")
    axes[1].set_title("Representation quality vs EMA momentum")
    for i, v in enumerate(results["linear_probe_acc"]):
        if v is not None:
            axes[1].text(i, v + 0.002, f"{v:.3f}", ha="center", fontsize=10)

    plt.tight_layout()
    plt.savefig("outputs/figures/ema_momentum_sweep.png", dpi=150)
    plt.show()

    print("Best momentum (FID):      ", results.loc[results["fid"].idxmin(), "teacher_momentum"])
    print("Best momentum (probe acc):", results.loc[results["linear_probe_acc"].idxmax(), "teacher_momentum"])
